In [1]:
import numpy as np
import pandas as pd
import torch
from pytabkit import RealMLP_TD_Classifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_score,
    recall_score,
    roc_auc_score,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

d:\Python\Predicting Credit Card\.venv\Lib\site-packages\torch\jit\_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
W1119 16:57:16.742000 32100 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Using device: cuda


In [6]:
# 1. Đọc dữ liệu train.csv (đã có cột fold chia từ StratifiedKFold)
print("Đang đọc dữ liệu train.csv...")
train = pd.read_csv("train.csv")

target_col = "is_fraud"
id_col = "id"
fold_col = "fold"

feature_cols = [c for c in train.columns if c not in [target_col, id_col, fold_col]]

# 2. Xử lý các cột boolean sang int/float (0 và 1)
bool_cols = train[feature_cols].select_dtypes(include=["bool"]).columns.tolist()
for col in bool_cols:
    train[col] = train[col].astype(float)

# 3. Dùng OrdinalEncoder mã hóa các cột categorical dạng text thành số (0, 1, 2, ...)
# Điều này giúp Scikit-learn check_array không bị lỗi 'Cannot cast str dtype to float64'
cat_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
train[cat_cols] = encoder.fit_transform(train[cat_cols].fillna("Missing").astype(str))

print(f"Số lượng features: {len(feature_cols)}")
print(f"Danh sách các folds: {sorted(train[fold_col].unique())}")

Đang đọc dữ liệu train.csv...
Số lượng features: 24
Danh sách các folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [ ]:
# 4. Huấn luyện 5 model RealMLP theo K-Fold
models = []
oof_preds = np.zeros(len(train))
fold_metrics = []

n_splits = train[fold_col].nunique()
print(f"Bắt đầu huấn luyện {n_splits}-Fold RealMLP (TD Classifier)...")

for fold in range(n_splits):
    print(f"\n{'='*20} FOLD {fold} {'='*20}")
    
    train_data = train[train[fold_col] != fold]
    val_data = train[train[fold_col] == fold]
    
    X_train, y_train = train_data[feature_cols], train_data[target_col].astype(int)
    X_val, y_val = val_data[feature_cols], val_data[target_col].astype(int)
    val_indices = val_data.index
    
    # Khởi tạo RealMLP-TD Classifier trên GPU
    model = RealMLP_TD_Classifier(
        device=device,
        random_state=42 + fold,
        verbosity=2,
    )
    
    print(f"Đang fit RealMLP Fold {fold}...")
    # Truyền cat_col_names để RealMLP nhận diện và áp dụng embedding riêng cho các cột này
    model.fit(
        X_train,
        y_train,
        X_val=X_val,
        y_val=y_val,
        cat_col_names=cat_cols,

    )
    models.append(model)
    
    # Dự đoán xác suất lớp fraud (class 1)
    val_prob = model.predict_proba(X_val)[:, 1]
    oof_preds[val_indices] = val_prob
    
    # Đánh giá 4 chỉ số trên fold hiện tại
    val_pred = (val_prob >= 0.5).astype(int)
    f_roc_auc = roc_auc_score(y_val, val_prob)
    f_auc_pr = average_precision_score(y_val, val_prob)
    f_prec = precision_score(y_val, val_pred, zero_division=0)
    f_rec = recall_score(y_val, val_pred, zero_division=0)
    
    fold_metrics.append({
        "fold": fold,
        "ROC-AUC": f_roc_auc,
        "AUC-PR": f_auc_pr,
        "Precision": f_prec,
        "Recall": f_rec
    })
    print(f"Fold {fold} - ROC-AUC: {f_roc_auc:.4f} | AUC-PR: {f_auc_pr:.4f} | Precision: {f_prec:.4f} | Recall: {f_rec:.4f}")

Bắt đầu huấn luyện 5-Fold RealMLP (TD Classifier)...

==================== FOLD 0 ====================
Đang fit RealMLP Fold 0...


d:\Python\Predicting Credit Card\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


SystemExit: 1

d:\Python\Predicting Credit Card\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# 5. Đánh giá tổng hợp trên toàn bộ Out-Of-Fold (OOF)
y_all = train[target_col].astype(int)
oof_pred_binary = (oof_preds >= 0.5).astype(int)

oof_roc_auc = roc_auc_score(y_all, oof_preds)
oof_auc_pr = average_precision_score(y_all, oof_preds)
oof_precision = precision_score(y_all, oof_pred_binary, zero_division=0)
oof_recall = recall_score(y_all, oof_pred_binary, zero_division=0)

print("\n" + "="*45)
print("--- KẾT QUẢ TỪNG FOLD ---")
print(pd.DataFrame(fold_metrics).to_string(index=False))

print("\n" + "="*45)
print("--- TỔNG KẾT ĐÁNH GIÁ OUT-OF-FOLD (OOF) REALMLP ---")
print(f"OOF ROC-AUC   : {oof_roc_auc:.4f}")
print(f"OOF AUC-PR    : {oof_auc_pr:.4f}")
print(f"OOF Precision : {oof_precision:.4f} (với threshold = 0.5)")
print(f"OOF Recall    : {oof_recall:.4f} (với threshold = 0.5)")
print("="*45)

print("\nBáo cáo chi tiết Classification Report OOF:")
print(classification_report(y_all, oof_pred_binary, digits=4))

In [ ]:
# 6. Dự đoán trên test.csv bằng cách lấy trung bình (Ensemble Averaging) từ 5 mô hình RealMLP
try:
    print("Đang đọc dữ liệu test.csv...")
    test = pd.read_csv("test.csv")
    X_test = test[feature_cols].copy()
    
    # Áp dụng chuyển đổi boolean và OrdinalEncoder giống hệt train
    for col in bool_cols:
        if col in X_test.columns:
            X_test[col] = X_test[col].astype(float)
            
    X_test[cat_cols] = encoder.transform(X_test[cat_cols].fillna("Missing").astype(str))
    
    print("Đang dự đoán kết quả test từ 5 models RealMLP...")
    test_preds = np.zeros(len(test))
    for i, model in enumerate(models):
        fold_pred = model.predict_proba(X_test)[:, 1]
        test_preds += fold_pred / len(models)
    
    submission = pd.DataFrame({
        id_col: test[id_col],
        target_col: test_preds
    })
    submission.to_csv("submission_realmlp_kfold.csv", index=False)
    print("Đã lưu kết quả dự đoán vào submission_realmlp_kfold.csv thành công!")
except Exception as e:
    print("Lỗi khi dự đoán test:", e)